# Tools
Models can request to call tools to perform task such as fetching data from database, searching the web, or running the code. Tools are pairings of: 
1. A schema, including the name of the tool,  the description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to exeute. 

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]= os.getenv("GOOGLE_API_KEY")


In [ ]:
from langchain.chat_models import init_chat_model

model= init_chat_model(
    "gemini-3.1-flash-lite",
     model_provider="google_genai",
     thinking_budget=0,
)
response = model.invoke("How does the sun glow")
response

## Creating a Tool
We we import the tool and use the decorator over the function to make it a tool. 

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get weather at a location"""
    return f"Its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What is the weather like in boston")
print(response)
for tool_call in response.tool_calls:
    # View tools calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "boston"}'}, '__gemini_function_call_thought_signatures__': {'call_5170798741210625809': 'EqEBCp4BAWkUfROYnLeHvstGWv0c1JYmSzG4fnecn/MxUa4S2HaunZibicBMW8rAWGA/b0mjvtw6AI0lXOonTAU37HTcAUTrpANy3MqcjZAm2VEwEAgQqwGdl19O0CQmdgrxFh5Ln1NWlfo+mIL2S5sHkwIDQoWN4j5+kYCnBUYz1KzHxxPZ8yhZE5yYvzZN7Km4ffy2X8TpVEPbLh1FrwAmrxo='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'} id='lc_run--01a1ff49-1060-72c4-99d5-f99e070f5e95-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': 'call_5170798741210625809', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 16, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Args: {'location': 'boston'}


: 

### Another Way to Use the tools

In [ ]:
from langchain.agents import create_agent

def get_weather(location:str)-> str:
    """Get weather at a location"""
    return f"Its Sunny in this {location}"

agent=create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
agent

In [9]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content)


[{'type': 'text', 'text': "It's currently sunny in San Francisco!", 'extras': {'signature': 'Eo4BCosBAWkUfROyrm63zlo5YoAd+1N5s0bp2fDdUQmz9CWYf32q9zBwNbhr2Yl04+0ruOryIrSXCw8NsHzqNVZBeWmBn4P+V7WkUhXPrbiWGIEIa6ScCEVc2lmCAr2ZVZWZ7c2jkKzatuk2ny4qEYTSxUHhaY6ZS2BJWcfYN53Z1GZeaR+r+bn9G0P0fGaQmw=='}}]
